## Sequential Workflow with MAF and Microsoft Foundry

![sequential-workflow](./Images/sequential-workflow.png)

### Installing Utilities and Libraries

In [ ]:
%pip install agent-framework==1.19.0 azure-ai-projects==2.7.0 azure-identity==1.25.3 azure-monitor-opentelemetry==1.8.8

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential

load_dotenv()
foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

credential = AzureCliCredential()

### Creating the Foundry Chat Client

In [ ]:
from agent_framework import Agent
from azure.ai.projects import AIProjectClient
from agent_framework.foundry import FoundryAgent, FoundryChatClient, to_prompt_agent
from azure.identity import AzureCliCredential

client = FoundryChatClient(
    project_endpoint = foundry_project_endpoint,
    model = model_deployment_name,
    credential = credential
)

await client.configure_azure_monitor(
    enable_sensitive_data=True
)

### Define the Web Search Tool

In [ ]:
web_search_tool = FoundryChatClient.get_web_search_tool()

### Creating the Researcher Agent

In [ ]:
researcher_agent = Agent(
    client=client,

    name="Researcher-Agent",

    instructions="""You are a knowledgeable researcher. Your task is to gather information and provide
    insights on a given topic. You should use reliable sources and present the information in a 
    clear and concise manner
    """,

    tools=[
        web_search_tool
    ]
)

### Creating the Writer Agent

In [ ]:
writer_agent = Agent(
    client=client,

    name="Writer-Agent",

    instructions=""" You are a create writer. Your task is to write an essay on a given topic.
    You should focus on clarity, coherence and engaging storytelling.
    """
)

### Creating the First Executor to run the Researcher Agent

In [ ]:
from agent_framework import WorkflowBuilder, WorkflowContext, executor
from typing import Any
from agent_framework import ChatResponse

@executor(id = "run_researcher_agent")
async def run_researcher_agent(query: str,
                               ctx: WorkflowContext[str]) -> None:
    response = await researcher_agent.run(query)

    await ctx.send_message(str(response))

### Creating the Second Executor to run the Writer Agent

In [ ]:
@executor(id = "run_writer_agent")
async def run_writer_agent(research_data: str,
                          ctx: WorkflowContext[str]) -> None:
    response = await writer_agent.run(research_data)

    await ctx.yield_output(str(response))

### Creating the Sequential Workflow

In [ ]:
from agent_framework import WorkflowBuilder, WorkflowViz

workflow = (
    WorkflowBuilder(
        start_executor=run_researcher_agent
    )
    .add_edge(
        run_researcher_agent,
        run_writer_agent
    )
    .build()
)

viz = WorkflowViz(workflow)

### Generate Mermaid Diagram for Visualization

In [ ]:
mermaid_content = viz.to_mermaid()

# printing mermaid content as markdown
from IPython.display import Markdown, display
display(Markdown(f"```mermaid\n{mermaid_content}\n```"))

### Run the Workflow and Stream Events

In [ ]:
async def main():

    async for event in workflow.run(
        "Write an essay about the impact of AI on society.",
        stream=True
    ):

        print(f"Event: {event}")

        if event.type == "output":
            print("\n=== Workflow Output ===\n")
            print(event.data)


await main()